# LLaMA 1B x MATH — W4A16 — All 50 Problems

**Quantization:** 4-bit NF4 weights via bitsandbytes

**Problems:** 50 total (20 Level 1, 20 Level 3, 10 Level 5)

**Strategies:** Greedy + Self-Consistency N=5

**Output:** `llama_math_w4a16.csv`

**Estimated time:** ~180-200 min on T4

In [ ]:
# ── CELL 1: Install ───────────────────────────────────────────────────────────
!pip install -q "transformers>=4.40.0" "datasets>=2.19.0" "accelerate>=0.29.0" "bitsandbytes>=0.43.0"
print('Done')

In [ ]:
# ── CELL 2: HuggingFace login — required for gated LLaMA ──────────────────────
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('HF_TOKEN')) 
print('Logged in via hardcoded token')

In [ ]:
# ── CELL 3: GPU check ─────────────────────────────────────────────────────────
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU — Runtime > Change runtime type > T4 GPU')
print(f'GPU:  {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# ── CELL 4: Imports ───────────────────────────────────────────────────────────
import re, gc, random, time, os
import torch, pandas as pd
from collections import Counter
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
print('Imports OK')

In [ ]:
# ── CELL 5: Config ────────────────────────────────────────────────────────────
MODEL_ID    = 'meta-llama/Llama-3.2-1B-Instruct'
QUANT_TYPE  = 'w4a16'
OUTPUT_PATH = '/content/llama_math_w4a16.csv'
SEED        = 42
MAX_NEW_TOKENS = 1024
SC_N        = 5
SC_TEMP     = 0.7

random.seed(SEED)
torch.manual_seed(SEED)
print(f'Quant: {QUANT_TYPE} | Output: {OUTPUT_PATH}')


In [ ]:
# ── CELL 6: Load MATH dataset — all 50 problems ───────────────────────────────
# Same dataset, seed, and sampling as BF16 runs so problem IDs match exactly

print('Loading MATH dataset...')
subjects = ['algebra', 'geometry', 'number_theory', 'prealgebra', 'precalculus']
datasets_list = []
for sub in subjects:
    print(f'  Loading {sub}...')
    datasets_list.append(load_dataset('EleutherAI/hendrycks_math', sub, split='test'))
ds_math = concatenate_datasets(datasets_list)
print(f'Total available: {len(ds_math)}')

# Same shuffle seed as BF16 — produces identical problem pool
random.seed(SEED)
easy_all, medium_all, hard_all = [], [], []
for i, row in enumerate(ds_math):
    level = row['level']
    if '1' in level:   easy_all.append(i)
    elif '3' in level: medium_all.append(i)
    elif '5' in level: hard_all.append(i)

random.shuffle(easy_all)
random.shuffle(medium_all)
random.shuffle(hard_all)

# Take all 50 problems — same 50 as BF16 runs
all_l1 = easy_all[:20]
all_l3 = medium_all[:20]
all_l5 = hard_all[:10]

problems = []
for idx in all_l1:
    row = ds_math[idx]
    problems.append({'pid': idx, 'difficulty': 'Level 1',
                     'question': row['problem'], 'answer': row['solution']})
for idx in all_l3:
    row = ds_math[idx]
    problems.append({'pid': idx, 'difficulty': 'Level 3',
                     'question': row['problem'], 'answer': row['solution']})
for idx in all_l5:
    row = ds_math[idx]
    problems.append({'pid': idx, 'difficulty': 'Level 5',
                     'question': row['problem'], 'answer': row['solution']})

print(f'Loaded {len(problems)} problems')
for diff in ['Level 1', 'Level 3', 'Level 5']:
    print(f'  {diff}: {sum(1 for p in problems if p["difficulty"]==diff)}')

In [ ]:
# ── CELL 7: Helpers ───────────────────────────────────────────────────────────

def extract_ground_truth(solution_str):
    matches = re.findall(r'\\boxed\{([^}]+)\}', solution_str)
    if matches: return matches[-1].strip()
    nums = re.findall(r'-?[\d\.]+', solution_str)
    return nums[-1] if nums else None

def extract_prediction(output_str):
    matches = re.findall(r'\\boxed\{([^}]+)\}', output_str)
    if matches: return matches[-1].strip()
    m = re.search(r'####\s*([\d,\.\-]+)', output_str)
    if m: return m.group(1).replace(',','').strip()
    nums = re.findall(r'\b([\d,\.]+)\b', output_str)
    if nums: return nums[-1].replace(',','').strip()
    return None

def format_prompt(question):
    return (
        '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n'
        'You are a math problem solver. Show your reasoning step by step. '
        'End your answer with the final result in \\boxed{}.\n'
        '<|eot_id|><|start_header_id|>user<|end_header_id|>\n'
        f'{question}\n'
        '<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n'
    )

def detect_first_error_step(text):
    steps = re.split(r'\n+', text)
    for step_num, step in enumerate(steps, 1):
        matches = re.findall(r'([\d\.]+)\s*([\+\-\*\/])\s*([\d\.]+)\s*=\s*([\d\.]+)', step)
        for a, op, b, result in matches:
            try:
                a, b, result = float(a), float(b), float(result)
                expected = {'+':a+b,'-':a-b,'*':a*b,'/':a/b if b!=0 else None}.get(op)
                if expected is not None and abs(expected-result) > 0.01:
                    return step_num
            except: continue
    return None

print('Helpers ready')

In [ ]:
# ── CELL 8: Load model W4A16 ──────────────────────────────────────────────────

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'left'

print('Loading model NF4 (W4A16)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16
    ),
    device_map='auto',
    trust_remote_code=True,
)
model.eval()
print(f'GPU mem: {torch.cuda.memory_allocated()/1e9:.2f} GB')


In [ ]:
# ── CELL 9: Run greedy — all 50 problems ──────────────────────────────────────
# Saves after every problem — resume-safe

existing    = pd.read_csv(OUTPUT_PATH) if os.path.exists(OUTPUT_PATH) else pd.DataFrame()
done_greedy = set(existing[existing['decoding_strategy']=='greedy']['pid'].tolist()) \
              if len(existing) > 0 else set()
greedy_results = []

print('='*55)
print(f'GREEDY — W4A16 — 50 problems')
print('='*55)
t0 = time.time()

for i, prob in enumerate(problems):
    if prob['pid'] in done_greedy:
        print(f'  [{i+1}/50] pid={prob["pid"]} skipping (already done)')
        continue

    gt     = extract_ground_truth(prob['answer'])
    inputs = tokenizer(format_prompt(prob['question']), return_tensors='pt').to(model.device)
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                                  do_sample=False, pad_token_id=tokenizer.eos_token_id)
    generated = tokenizer.decode(out_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    pred    = extract_prediction(generated)
    correct = (pred is not None and gt is not None and pred.strip()==gt.strip())
    err     = detect_first_error_step(generated) if not correct else None
    elapsed = (time.time()-t0)/60

    greedy_results.append({
        'quant_type': QUANT_TYPE, 'decoding_strategy': 'greedy',
        'pid': prob['pid'], 'difficulty': prob['difficulty'],
        'predicted_answer': pred, 'ground_truth': gt, 'correct': correct,
        'reasoning_length_tokens': len(out_ids[0])-inputs['input_ids'].shape[1],
        'first_error_step': err, 'raw_output': generated[:500], 'n_samples': None,
    })
    pd.concat([existing, pd.DataFrame(greedy_results)], ignore_index=True).to_csv(OUTPUT_PATH, index=False)

    status = 'CORRECT' if correct else 'wrong'
    print(f'  [{i+1}/50] pid={prob["pid"]:<6} {prob["difficulty"]:<8} '
          f'pred={str(pred):<12} gt={str(gt):<12} {status}  {elapsed:.0f}min')

n_correct = sum(r['correct'] for r in greedy_results)
print('='*55)
print(f'Greedy done: {n_correct}/{len(greedy_results)} ({100*n_correct/max(len(greedy_results),1):.1f}%)  {(time.time()-t0)/60:.1f}min')

In [ ]:
# ── CELL 10: Run SC N=5 — all 50 problems ─────────────────────────────────────
# Saves after every problem

existing2 = pd.read_csv(OUTPUT_PATH) if os.path.exists(OUTPUT_PATH) else pd.DataFrame()
done_sc   = set(existing2[existing2['decoding_strategy']=='self_consistency']['pid'].tolist()) \
            if len(existing2) > 0 else set()
sc_results = []

print('='*55)
print(f'SELF-CONSISTENCY N=5 — W4A16 — 50 problems')
print('='*55)
t0 = time.time()

for i, prob in enumerate(problems):
    if prob['pid'] in done_sc:
        print(f'  [{i+1}/50] pid={prob["pid"]} skipping')
        continue

    gt     = extract_ground_truth(prob['answer'])
    inputs = tokenizer(format_prompt(prob['question']), return_tensors='pt').to(model.device)
    path_preds = []
    for _ in range(SC_N):
        with torch.no_grad():
            out_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                                      do_sample=True, temperature=SC_TEMP,
                                      pad_token_id=tokenizer.eos_token_id)
        gen = tokenizer.decode(out_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        p = extract_prediction(gen)
        if p is not None: path_preds.append(p.strip())

    final_pred = Counter(path_preds).most_common(1)[0][0] if path_preds else None
    correct    = (final_pred is not None and gt is not None and final_pred.strip()==gt.strip())
    elapsed    = (time.time()-t0)/60

    sc_results.append({
        'quant_type': QUANT_TYPE, 'decoding_strategy': 'self_consistency',
        'pid': prob['pid'], 'difficulty': prob['difficulty'],
        'predicted_answer': final_pred, 'ground_truth': gt, 'correct': correct,
        'reasoning_length_tokens': SC_N, 'first_error_step': None,
        'raw_output': str(path_preds), 'n_samples': SC_N,
    })
    pd.concat([existing2, pd.DataFrame(sc_results)], ignore_index=True).to_csv(OUTPUT_PATH, index=False)

    status = 'CORRECT' if correct else 'wrong'
    print(f'  [{i+1}/50] pid={prob["pid"]:<6} {prob["difficulty"]:<8} '
          f'vote={str(final_pred):<12} gt={str(gt):<12} {status}  paths={len(path_preds)}/{SC_N}  {elapsed:.0f}min')

n_correct_sc = sum(r['correct'] for r in sc_results)
print('='*55)
print(f'SC done: {n_correct_sc}/{len(sc_results)} ({100*n_correct_sc/max(len(sc_results),1):.1f}%)  {(time.time()-t0)/60:.1f}min')

In [ ]:
# ── CELL 11: Summary + save to Drive ──────────────────────────────────────────
final_df = pd.read_csv(OUTPUT_PATH)
print(f'Total rows: {len(final_df)}')
for strat in final_df['decoding_strategy'].unique():
    sub = final_df[final_df['decoding_strategy']==strat]
    print(f'{strat}: {sub["correct"].mean()*100:.1f}% ({int(sub["correct"].sum())}/{len(sub)})')
    for diff in ['Level 1','Level 3','Level 5']:
        d = sub[sub['difficulty']==diff]
        if len(d)>0:
            print(f'  {diff}: {d["correct"].mean()*100:.1f}% ({int(d["correct"].sum())}/{len(d)})')

# Save to Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import shutil
drive_path = '/content/drive/MyDrive/quantization_research/llama_math/llama_math_w4a16.csv'
os.makedirs(os.path.dirname(drive_path), exist_ok=True)
shutil.copy(OUTPUT_PATH, drive_path)
print(f'\nSaved to Drive: {drive_path}')
print('Also download from Files panel as backup')

In [ ]:
# ── CELL 12: Cleanup ──────────────────────────────────────────────────────────
del model
gc.collect()
torch.cuda.empty_cache()
print(f'GPU mem: {torch.cuda.memory_allocated()/1e9:.2f} GB')